In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt


# =============================================================================
# Task 4 — Recode SMOD L2 to SMOD L1 (3 classes)
# Outputs (all prefixed with task4_):
#   outputs/ : GeoTIFF + JSON
#   tables/  : CSV
#   figures/ : PNG
# =============================================================================

# ----------------------------
# Project structure
# ----------------------------
ROOT = Path(r"C:\Users\am636\copperbelt_worldpop_smod")

DATA = ROOT / "data_raw"
OUT = ROOT / "outputs"
TABDIR = ROOT / "tables"
FIGDIR = ROOT / "figures"

for d in (DATA, OUT, TABDIR, FIGDIR):
    d.mkdir(parents=True, exist_ok=True)

PREFIX = "task4_"

smod_l2_path = DATA / "GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif"

missing = [p for p in (smod_l2_path,) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input file in data_raw:\n" + "\n".join(str(p) for p in missing))

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# ----------------------------
# Output paths
# ----------------------------
out_l1 = OUT / f"{PREFIX}smod_L1_from_L2_54009_1km_INT16.tif"
out_counts = TABDIR / f"{PREFIX}smod_L1_value_counts.csv"
out_png = FIGDIR / f"{PREFIX}smod_L1_preview.png"
out_report = OUT / f"{PREFIX}smod_L1_recode_report.json"


# ----------------------------
# Recode settings
# ----------------------------
NODATA_OUT = -200
EXPECTED_L2 = {10, 11, 12, 13, 21, 22, 23, 30}


def recode_l2_to_l1_int16(arr_l2: np.ndarray, nodata_in: int) -> tuple[np.ndarray, dict]:
    """
    Recode L2 categorical values to L1 categorical values (Int16).
    Rules:
      - 11,12,13 -> 1
      - 21,22,23 -> 2
      - 30       -> 3
      - 10 (water) and nodata_in -> NODATA_OUT
      - Any other values -> NODATA_OUT (reported)
    """
    out = np.full(arr_l2.shape, NODATA_OUT, dtype=np.int16)

    valid = arr_l2 != nodata_in

    out[valid & np.isin(arr_l2, [11, 12, 13])] = 1
    out[valid & np.isin(arr_l2, [21, 22, 23])] = 2
    out[valid & (arr_l2 == 30)] = 3

    unknown_mask = valid & (~np.isin(arr_l2, list(EXPECTED_L2)))
    unknown_count = int(unknown_mask.sum())
    unknown_values = np.unique(arr_l2[unknown_mask]).tolist() if unknown_count > 0 else []

    water_count = int((valid & (arr_l2 == 10)).sum())

    stats = {
        "water_cells_set_to_nodata": water_count,
        "unknown_cells_set_to_nodata": unknown_count,
        "unknown_values": unknown_values,
    }
    return out, stats


# ----------------------------
# 1) Read L2 raster
# ----------------------------
with rasterio.open(smod_l2_path) as src:
    l2 = src.read(1)
    profile_in = src.profile.copy()
    crs_in = src.crs
    nodata_in = src.nodata
    dtype_in = src.dtypes[0]

if nodata_in is None:
    nodata_in = -200

print("Input:")
print(" -", smod_l2_path.name)
print(" - dtype:", dtype_in, "| nodata:", nodata_in, "| CRS:", crs_in, "| shape:", l2.shape)
print()


# ----------------------------
# 2) Recode to L1 (Int16)
# ----------------------------
l1, recode_stats = recode_l2_to_l1_int16(l2.astype(np.int16, copy=False), int(nodata_in))

if l1.dtype != np.int16:
    raise TypeError("Output is not Int16.")

vals, counts = np.unique(l1, return_counts=True)
count_tbl = pd.DataFrame({"value": vals.astype(int), "cell_count": counts.astype(int)})
count_tbl["class"] = count_tbl["value"].map({
    -200: "NoData (incl. Water)",
    1: "Rural (1)",
    2: "Urban cluster (2)",
    3: "Urban centre (3)",
}).fillna("Other/Unexpected")
count_tbl.to_csv(out_counts, index=False)

print("Recode stats:", recode_stats)
print("Saved:", out_counts)
print()


# ----------------------------
# 3) Write L1 raster (Int16, NoData=-200)
# ----------------------------
profile_out = profile_in.copy()
profile_out.update(
    dtype=rasterio.int16,
    nodata=NODATA_OUT,
    count=1,
    compress="LZW",
)

with rasterio.open(out_l1, "w", **profile_out) as dst:
    dst.write(l1, 1)

print("Saved:", out_l1)
print()


# ----------------------------
# 4) Preview image (display-only masking)
# ----------------------------
preview = l1.astype(np.float32)
preview[preview == NODATA_OUT] = np.nan

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(preview, interpolation="nearest")
ax.set_title("SMOD L1 (from L2): 1=Rural, 2=Urban cluster, 3=Urban centre; NoData=-200")
ax.set_axis_off()
cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_ticks([1, 2, 3])
cbar.set_ticklabels(["Rural (1)", "Urban cluster (2)", "Urban centre (3)"])
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_png)
print()


# ----------------------------
# 5) JSON report
# ----------------------------
report = {
    "run_time_utc": run_time_utc,
    "input": {
        "path": str(smod_l2_path),
        "dtype": str(dtype_in),
        "nodata": int(nodata_in) if nodata_in is not None else None,
        "crs": str(crs_in),
        "shape": [int(l2.shape[0]), int(l2.shape[1])],
    },
    "output": {
        "path": str(out_l1),
        "dtype": "int16",
        "nodata": NODATA_OUT,
        "crs": str(crs_in),
        "shape": [int(l1.shape[0]), int(l1.shape[1])],
    },
    "recode": {
        "mapping": {
            "10": "NoData (-200)",
            "11,12,13": "1",
            "21,22,23": "2",
            "30": "3",
            "input NoData": "NoData (-200)",
        },
        "stats": recode_stats,
    },
    "scope": "Recode only (no reprojection/resampling/harmonisation).",
}

with open(out_report, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved:", out_report)
print()


# ----------------------------
# 6) Short run note (console)
# ----------------------------
print("Task 4 note:")
print("Reclassified SMOD L2 -> L1 (3 classes) as Int16, set water (10) to NoData, preserved NoData as -200.")


SMOD L2 input:
 - path: GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0_R11_C21.tif
 - dtype: int16
 - nodata: -200.0
 - CRS: ESRI:54009
 - shape: (1000, 1000)

Recode stats: {'water_cells_set_to_nodata': 10263, 'unknown_cells_set_to_nodata': 0, 'unknown_values': []}
Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_value_counts.csv

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_from_L2_54009_1km_INT16.tif

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_preview.png

Wrote: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\04_smod_L1_recode_report.json

--- Interview notes (Task 4 — recode only) ---
Created SMOD L1 by reclassifying L2 codes (ca